# Credit Risk Predictive Modeling

## Modeling Objective

The objective is to estimate the probability of next-month default using historical credit and repayment behavior.

The primary model focuses on financial and behavioral attributes rather than demographic variables.

Two models will be evaluated:

1. Logistic Regression — interpretable baseline model
2. Random Forest — nonlinear comparison model

Because the target population is imbalanced, model performance will be evaluated using multiple metrics including ROC-AUC, precision, recall, F1-score, and the confusion matrix rather than accuracy alone.

In [3]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

PROJECT_ROOT = Path.cwd().parent

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_default_processed.csv"
)

df = pd.read_csv(DATA_FILE)

print("Dataset shape:", df.shape)
print("Default rate:", f"{df['default_next_month'].mean():.2%}")

df.head()

Dataset shape: (30000, 28)
Default rate: 22.12%


,customer_id,credit_limit,sex,education,marital_status,age,default_next_month,sex_code,education_code,marital_status_code,...,bill_amount_jul,bill_amount_jun,bill_amount_may,bill_amount_apr,payment_amount_sep,payment_amount_aug,payment_amount_jul,payment_amount_jun,payment_amount_may,payment_amount_apr
0,1,20000,Female,University,Married,24,1,2,2,1,...,689,0,0,0,0,689,0,0,0,0
1,2,120000,Female,University,Single,26,1,2,2,2,...,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,3,90000,Female,University,Single,34,0,2,2,2,...,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,4,50000,Female,University,Married,37,0,2,2,1,...,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,5,50000,Male,University,Married,57,0,1,2,1,...,35835,20940,19146,19131,2000,36681,10000,9000,689,679


In [4]:
feature_columns = [
    "credit_limit",
    "repayment_status_sep",
    "repayment_status_aug",
    "repayment_status_jul",
    "repayment_status_jun",
    "repayment_status_may",
    "repayment_status_apr",
    "bill_amount_sep",
    "bill_amount_aug",
    "bill_amount_jul",
    "bill_amount_jun",
    "bill_amount_may",
    "bill_amount_apr",
    "payment_amount_sep",
    "payment_amount_aug",
    "payment_amount_jul",
    "payment_amount_jun",
    "payment_amount_may",
    "payment_amount_apr",
]

X = df[feature_columns].copy()
y = df["default_next_month"].copy()

print("Model features:", len(feature_columns))
print("X shape:", X.shape)
print("y shape:", y.shape)

Model features: 19
X shape: (30000, 19)
y shape: (30000,)


In [5]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

PROJECT_ROOT = Path.cwd().parent

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_default_processed.csv"
)

df = pd.read_csv(DATA_FILE)

print("Dataset shape:", df.shape)
print("Default rate:", f"{df['default_next_month'].mean():.2%}")

df.head()

Dataset shape: (30000, 28)
Default rate: 22.12%


,customer_id,credit_limit,sex,education,marital_status,age,default_next_month,sex_code,education_code,marital_status_code,...,bill_amount_jul,bill_amount_jun,bill_amount_may,bill_amount_apr,payment_amount_sep,payment_amount_aug,payment_amount_jul,payment_amount_jun,payment_amount_may,payment_amount_apr
0,1,20000,Female,University,Married,24,1,2,2,1,...,689,0,0,0,0,689,0,0,0,0
1,2,120000,Female,University,Single,26,1,2,2,2,...,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,3,90000,Female,University,Single,34,0,2,2,2,...,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,4,50000,Female,University,Married,37,0,2,2,1,...,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,5,50000,Male,University,Married,57,0,1,2,1,...,35835,20940,19146,19131,2000,36681,10000,9000,689,679


In [6]:
feature_columns = [
    "credit_limit",

    "repayment_status_sep",
    "repayment_status_aug",
    "repayment_status_jul",
    "repayment_status_jun",
    "repayment_status_may",
    "repayment_status_apr",

    "bill_amount_sep",
    "bill_amount_aug",
    "bill_amount_jul",
    "bill_amount_jun",
    "bill_amount_may",
    "bill_amount_apr",

    "payment_amount_sep",
    "payment_amount_aug",
    "payment_amount_jul",
    "payment_amount_jun",
    "payment_amount_may",
    "payment_amount_apr",
]

X = df[feature_columns].copy()
y = df["default_next_month"].copy()

print("Model features:", len(feature_columns))
print("X shape:", X.shape)
print("y shape:", y.shape)

Model features: 19
X shape: (30000, 19)
y shape: (30000,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print(
    "Training default rate:",
    f"{y_train.mean():.2%}",
)

print(
    "Testing default rate:",
    f"{y_test.mean():.2%}",
)

Training records: 24000
Testing records: 6000
Training default rate: 22.12%
Testing default rate: 22.12%


## 3. Logistic Regression Baseline Model

Logistic Regression is used as the baseline model because it is widely understood, produces probability estimates, and provides a relatively interpretable benchmark for credit-risk classification.

The model standardizes numeric features before fitting.

In [8]:
logistic_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42,
            ),
        ),
    ]
)

logistic_model.fit(
    X_train,
    y_train,
)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [9]:
logistic_predictions = logistic_model.predict(X_test)

logistic_probabilities = (
    logistic_model.predict_proba(X_test)[:, 1]
)

print("Predictions created successfully.")

Predictions created successfully.


In [10]:
logistic_metrics = {
    "Accuracy": accuracy_score(
        y_test,
        logistic_predictions,
    ),
    "Precision": precision_score(
        y_test,
        logistic_predictions,
    ),
    "Recall": recall_score(
        y_test,
        logistic_predictions,
    ),
    "F1 Score": f1_score(
        y_test,
        logistic_predictions,
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        logistic_probabilities,
    ),
}

logistic_metrics_df = pd.DataFrame(
    logistic_metrics.items(),
    columns=[
        "Metric",
        "Value",
    ],
)

logistic_metrics_df["Value"] = (
    logistic_metrics_df["Value"]
    .round(4)
)

logistic_metrics_df

,Metric,Value
0,Accuracy,0.8093
1,Precision,0.7011
2,Recall,0.2404
3,F1 Score,0.3580
4,ROC-AUC,0.7064


In [11]:
print(
    classification_report(
        y_test,
        logistic_predictions,
        target_names=[
            "No Default",
            "Default",
        ],
    )
)

              precision    recall  f1-score   support

  No Default       0.82      0.97      0.89      4673
     Default       0.70      0.24      0.36      1327

    accuracy                           0.81      6000
   macro avg       0.76      0.61      0.62      6000
weighted avg       0.79      0.81      0.77      6000



### Baseline Model Interpretation

The Logistic Regression model achieves approximately 81% overall accuracy and a ROC-AUC of about 0.71, indicating moderate ability to distinguish between defaulting and non-defaulting clients.

However, recall for the Default class is relatively low at approximately 24%. This means that, using the standard 0.50 probability threshold, the model identifies only about one-quarter of actual defaults.

The result highlights an important credit-risk tradeoff: a model can achieve high overall accuracy while still failing to identify a large share of higher-risk clients. Threshold selection and additional model comparisons are therefore necessary.

In [12]:
conf_matrix = confusion_matrix(
    y_test,
    logistic_predictions,
)

confusion_df = pd.DataFrame(
    conf_matrix,
    index=[
        "Actual No Default",
        "Actual Default",
    ],
    columns=[
        "Predicted No Default",
        "Predicted Default",
    ],
)

confusion_df

,Predicted No Default,Predicted Default
Actual No Default,4537,136
Actual Default,1008,319


In [13]:
threshold_results = []

for threshold in [
    0.20,
    0.30,
    0.35,
    0.40,
    0.50,
]:
    predictions_at_threshold = (
        logistic_probabilities >= threshold
    ).astype(int)

    threshold_results.append(
        {
            "Threshold": threshold,
            "Accuracy": accuracy_score(
                y_test,
                predictions_at_threshold,
            ),
            "Precision": precision_score(
                y_test,
                predictions_at_threshold,
            ),
            "Recall": recall_score(
                y_test,
                predictions_at_threshold,
            ),
            "F1 Score": f1_score(
                y_test,
                predictions_at_threshold,
            ),
        }
    )

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df = threshold_df.round(4)

threshold_df

,Threshold,Accuracy,Precision,Recall,F1 Score
0,0.20,0.5968,0.3125,0.6858,0.4293
1,0.30,0.8012,0.5653,0.4371,0.4930
2,0.35,0.8077,0.5941,0.4115,0.4862
3,0.40,0.8122,0.6232,0.3813,0.4731
4,0.50,0.8093,0.7011,0.2404,0.3580


### Probability Threshold Tradeoff

The standard 0.50 classification threshold produces relatively high precision but identifies only about 24% of actual defaults.

Lowering the threshold improves default detection at the cost of additional false positives.

For demonstration purposes, a 0.30 threshold provides a more balanced result, with approximately 80% overall accuracy, 57% precision, and 44% recall for the Default class.

In a production credit-risk environment, the operating threshold would be selected based on business objectives, loss severity, risk appetite, review capacity, and the relative cost of false negatives versus false positives.

In [14]:
import plotly.express as px

threshold_long = threshold_df.melt(
    id_vars="Threshold",
    value_vars=[
        "Precision",
        "Recall",
        "F1 Score",
    ],
    var_name="Metric",
    value_name="Score",
)

fig = px.line(
    threshold_long,
    x="Threshold",
    y="Score",
    color="Metric",
    markers=True,
    title="Classification Performance by Probability Threshold",
    labels={
        "Threshold": "Probability Threshold",
        "Score": "Metric Score",
    },
)

fig.show()

In [15]:
from sklearn.ensemble import RandomForestClassifier

## 4. Random Forest Comparison Model

A Random Forest model is used as a nonlinear comparison to the Logistic Regression baseline.

The purpose is not simply to maximize accuracy, but to determine whether a more flexible model improves discrimination and default identification.

In [16]:
random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

random_forest_model.fit(
    X_train,
    y_train,
)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [17]:
rf_predictions = random_forest_model.predict(X_test)

rf_probabilities = (
    random_forest_model.predict_proba(X_test)[:, 1]
)

In [18]:
rf_metrics = {
    "Accuracy": accuracy_score(
        y_test,
        rf_predictions,
    ),
    "Precision": precision_score(
        y_test,
        rf_predictions,
    ),
    "Recall": recall_score(
        y_test,
        rf_predictions,
    ),
    "F1 Score": f1_score(
        y_test,
        rf_predictions,
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        rf_probabilities,
    ),
}

rf_metrics_df = pd.DataFrame(
    rf_metrics.items(),
    columns=[
        "Metric",
        "Value",
    ],
)

rf_metrics_df["Value"] = (
    rf_metrics_df["Value"]
    .round(4)
)

rf_metrics_df

,Metric,Value
0,Accuracy,0.7737
1,Precision,0.4904
2,Recall,0.5968
3,F1 Score,0.5384
4,ROC-AUC,0.7752


### Model Interpretation

The Random Forest model improves risk discrimination relative to the Logistic Regression baseline.

Its ROC-AUC increases to approximately 0.78, while recall for the Default class rises to approximately 60%. This means the model identifies substantially more actual defaults, although this improvement comes with lower precision and overall accuracy.

For this portfolio analysis, Random Forest is therefore treated as the stronger predictive model, while Logistic Regression remains an important interpretable benchmark.

In a production credit-risk environment, model selection would also consider calibration, stability, explainability, governance requirements, validation results, and the business cost of false positives and false negatives.

In [19]:
logistic_030_predictions = (
    logistic_probabilities >= 0.30
).astype(int)

model_comparison = pd.DataFrame(
    [
        {
            "Model": "Logistic Regression - 0.50 Threshold",
            "Accuracy": accuracy_score(
                y_test,
                logistic_predictions,
            ),
            "Precision": precision_score(
                y_test,
                logistic_predictions,
            ),
            "Recall": recall_score(
                y_test,
                logistic_predictions,
            ),
            "F1 Score": f1_score(
                y_test,
                logistic_predictions,
            ),
            "ROC-AUC": roc_auc_score(
                y_test,
                logistic_probabilities,
            ),
        },
        {
            "Model": "Logistic Regression - 0.30 Threshold",
            "Accuracy": accuracy_score(
                y_test,
                logistic_030_predictions,
            ),
            "Precision": precision_score(
                y_test,
                logistic_030_predictions,
            ),
            "Recall": recall_score(
                y_test,
                logistic_030_predictions,
            ),
            "F1 Score": f1_score(
                y_test,
                logistic_030_predictions,
            ),
            "ROC-AUC": roc_auc_score(
                y_test,
                logistic_probabilities,
            ),
        },
        {
            "Model": "Random Forest",
            "Accuracy": accuracy_score(
                y_test,
                rf_predictions,
            ),
            "Precision": precision_score(
                y_test,
                rf_predictions,
            ),
            "Recall": recall_score(
                y_test,
                rf_predictions,
            ),
            "F1 Score": f1_score(
                y_test,
                rf_predictions,
            ),
            "ROC-AUC": roc_auc_score(
                y_test,
                rf_probabilities,
            ),
        },
    ]
)

numeric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC",
]

model_comparison[numeric_columns] = (
    model_comparison[numeric_columns]
    .round(4)
)

model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression - 0.50 Threshold,0.8093,0.7011,0.2404,0.3580,0.7064
1,Logistic Regression - 0.30 Threshold,0.8012,0.5653,0.4371,0.4930,0.7064
2,Random Forest,0.7737,0.4904,0.5968,0.5384,0.7752


In [20]:
feature_importance_df = pd.DataFrame(
    {
        "Feature": feature_columns,
        "Importance": random_forest_model.feature_importances_,
    }
)

feature_importance_df = (
    feature_importance_df
    .sort_values(
        "Importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

feature_importance_df

,Feature,Importance
0,repayment_status_sep,0.275323
1,repayment_status_aug,0.124625
2,repayment_status_jul,0.072548
3,repayment_status_jun,0.058176
4,credit_limit,0.050273
5,payment_amount_aug,0.040514
6,bill_amount_sep,0.040367
7,payment_amount_sep,0.038569
8,repayment_status_may,0.038503
9,payment_amount_jul,0.034849


In [21]:
fig = px.bar(
    feature_importance_df.head(10),
    x="Importance",
    y="Feature",
    orientation="h",
    title="Top 10 Random Forest Feature Importances",
    labels={
        "Importance": "Feature Importance",
        "Feature": "Risk Feature",
    },
)

fig.update_layout(
    yaxis={
        "categoryorder": "total ascending"
    }
)

fig.show()

### Feature Importance Interpretation

Recent repayment behavior is the strongest source of predictive information in the Random Forest model.

The most recent repayment-status variable is the highest-ranked feature, followed by repayment behavior from prior months. Credit limit and recent billing/payment amounts also contribute to the model, but to a lesser extent.

These results are consistent with the exploratory analysis, which showed substantially higher default rates among clients with recent and repeated delinquency.

Random Forest feature importance measures predictive contribution rather than causal impact or direction of effect. Business interpretation should therefore be supported by the underlying exploratory analysis.

In [22]:
from sklearn.metrics import roc_curve

In [23]:
logistic_fpr, logistic_tpr, _ = roc_curve(
    y_test,
    logistic_probabilities,
)

rf_fpr, rf_tpr, _ = roc_curve(
    y_test,
    rf_probabilities,
)

roc_df = pd.concat(
    [
        pd.DataFrame(
            {
                "False Positive Rate": logistic_fpr,
                "True Positive Rate": logistic_tpr,
                "Model": "Logistic Regression",
            }
        ),
        pd.DataFrame(
            {
                "False Positive Rate": rf_fpr,
                "True Positive Rate": rf_tpr,
                "Model": "Random Forest",
            }
        ),
    ],
    ignore_index=True,
)

fig = px.line(
    roc_df,
    x="False Positive Rate",
    y="True Positive Rate",
    color="Model",
    title="ROC Curve — Model Comparison",
)

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=1,
    y1=1,
    line=dict(
        dash="dash",
    ),
)

fig.update_layout(
    xaxis_range=[0, 1],
    yaxis_range=[0, 1],
)

fig.show()

In [24]:
scored_test_df = X_test.copy()

scored_test_df["actual_default"] = (
    y_test.values
)

scored_test_df["default_risk_score"] = (
    rf_probabilities
)

scored_test_df["risk_band"] = pd.cut(
    scored_test_df["default_risk_score"],
    bins=[
        0,
        0.20,
        0.40,
        0.60,
        0.80,
        1.00,
    ],
    labels=[
        "Low",
        "Moderate",
        "Elevated",
        "High",
        "Very High",
    ],
    include_lowest=True,
)

risk_band_summary = (
    scored_test_df
    .groupby(
        "risk_band",
        observed=True,
    )
    .agg(
        clients=("actual_default", "count"),
        actual_defaults=("actual_default", "sum"),
        actual_default_rate=("actual_default", "mean"),
        average_model_score=("default_risk_score", "mean"),
    )
    .reset_index()
)

risk_band_summary["actual_default_rate_pct"] = (
    risk_band_summary["actual_default_rate"] * 100
).round(2)

risk_band_summary["average_model_score_pct"] = (
    risk_band_summary["average_model_score"] * 100
).round(2)

risk_band_summary

,risk_band,clients,actual_defaults,actual_default_rate,average_model_score,actual_default_rate_pct,average_model_score_pct
0,Low,578,23,0.039792,0.168268,3.98,16.83
1,Moderate,2949,358,0.121397,0.301797,12.14,30.18
2,Elevated,1280,270,0.210938,0.476909,21.09,47.69
3,High,619,281,0.453958,0.699102,45.40,69.91
4,Very High,574,395,0.688153,0.870424,68.82,87.04


In [25]:
fig = px.bar(
    risk_band_summary,
    x="risk_band",
    y="actual_default_rate_pct",
    text="actual_default_rate_pct",
    title="Observed Default Rate by Model Risk Band",
    labels={
        "risk_band": "Model Risk Band",
        "actual_default_rate_pct": "Observed Default Rate (%)",
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(
    yaxis_range=[
        0,
        risk_band_summary[
            "actual_default_rate_pct"
        ].max() * 1.15,
    ]
)

fig.show()

## 5. Modeling Conclusions

The modeling analysis demonstrates meaningful separation between lower- and higher-risk clients.

Random Forest provides stronger discrimination than the Logistic Regression baseline, achieving a ROC-AUC of approximately 0.78 and materially higher recall for actual defaults.

The model's strongest predictive features are recent repayment-status variables, which is consistent with the exploratory analysis showing substantially higher default rates among clients with recent and repeated delinquency.

Model-generated risk bands also show strong rank ordering. Observed default rates increase from approximately 4% in the Low-risk band to approximately 69% in the Very High-risk band.

These model scores are used for portfolio analytics and demonstration purposes only. They have not undergone formal probability calibration, independent model validation, regulatory approval, or production governance and should not be interpreted as production probability-of-default estimates.